In [ ]:
# %% [markdown]
# # All-cells UMAP colored by dominant gene program
#
# Each cell is colored by its highest-scoring program from the `program_order.txt` list.
#
# This version loads cNMF scores from `mdata`, but uses `X_umap` from the integrated AnnData object (`adata_int`). The notebook checks that `cnmf.obs_names` and `adata_int.obs_names` match, then reorders the integrated UMAP coordinates to the cNMF cell order before plotting.
# %%
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# %%
def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root by looking upward for expected project files."""
    start = start.resolve()
    required_paths = [
        Path("data/program_activity_long_v4.csv"),
        Path("analysis/regulator_dotplots/program_order.txt"),
    ]

    for candidate in [start, *start.parents]:
        if all((candidate / rel_path).exists() for rel_path in required_paths):
            return candidate

    raise FileNotFoundError(
        "Could not auto-detect REPO_ROOT. Set REPO_ROOT manually in this cell."
    )


# In a notebook, __file__ is usually not defined, so use an auto-detected repo root.
# If auto-detection fails, replace this with: REPO_ROOT = Path("/path/to/your/repo")
REPO_ROOT = find_repo_root()

OUT_DIR = REPO_ROOT / "analysis" / "regulator_dotplots" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FONT_DIR = REPO_ROOT / ".pixi" / "envs" / "default" / "fonts"
if FONT_DIR.exists():
    for font_file in fm.findSystemFonts(fontpaths=[str(FONT_DIR)]):
        fm.fontManager.addfont(font_file)

mpl.rcParams.update(
    {
        "font.family": "Arial",
        "font.size": 7,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        # Keep text editable in Illustrator when saving SVG.
        "svg.fonttype": "none",
    }
)

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"OUT_DIR   = {OUT_DIR}")

# %%
def align_umap_to_cnmf_cells(
    *,
    cnmf,
    adata_int,
    obsm_key: str = "X_umap",
    require_exact_match: bool = True,
) -> np.ndarray:
    """Return adata_int.obsm[obsm_key] reordered to cnmf.obs_names.

    The plotting code uses cNMF scores, so the UMAP coordinate rows must be in
    the same order as cnmf.obs_names. This function verifies that mapping.
    """
    cnmf_obs = pd.Index(cnmf.obs_names, name="cnmf_obs_names")
    adata_obs = pd.Index(adata_int.obs_names, name="adata_int_obs_names")

    if not cnmf_obs.is_unique:
        duplicated = cnmf_obs[cnmf_obs.duplicated()].unique()[:10].tolist()
        raise ValueError(f"cnmf.obs_names contains duplicate cell IDs, e.g. {duplicated}")

    if not adata_obs.is_unique:
        duplicated = adata_obs[adata_obs.duplicated()].unique()[:10].tolist()
        raise ValueError(f"adata_int.obs_names contains duplicate cell IDs, e.g. {duplicated}")

    if obsm_key not in adata_int.obsm_keys():
        raise KeyError(
            f"{obsm_key!r} was not found in adata_int.obsm. "
            f"Available obsm keys: {list(adata_int.obsm_keys())}"
        )

    missing_in_adata = cnmf_obs[~cnmf_obs.isin(adata_obs)]
    extra_in_adata = adata_obs[~adata_obs.isin(cnmf_obs)]

    if len(missing_in_adata) > 0:
        raise ValueError(
            "Some cNMF cells are missing from adata_int. "
            f"n_missing={len(missing_in_adata)}; first few: {missing_in_adata[:10].tolist()}"
        )

    if require_exact_match and len(extra_in_adata) > 0:
        raise ValueError(
            "adata_int contains cells that are not present in cNMF. "
            f"n_extra={len(extra_in_adata)}; first few: {extra_in_adata[:10].tolist()}"
        )

    adata_umap = np.asarray(adata_int.obsm[obsm_key])
    if adata_umap.ndim != 2 or adata_umap.shape[1] < 2:
        raise ValueError(
            f"adata_int.obsm[{obsm_key!r}] should be a 2D array with at least 2 columns; "
            f"got shape {adata_umap.shape}"
        )

    if adata_umap.shape[0] != adata_int.n_obs:
        raise ValueError(
            f"adata_int.obsm[{obsm_key!r}] has {adata_umap.shape[0]} rows, "
            f"but adata_int has {adata_int.n_obs} cells"
        )

    if cnmf_obs.equals(adata_obs):
        print("Cell indexes match in the same order; using adata_int X_umap directly.")
        aligned_umap = adata_umap[:, :2]
    else:
        print("Cell indexes match as IDs but not in the same order; reordering adata_int X_umap to cnmf.obs_names.")
        adata_positions = adata_obs.get_indexer(cnmf_obs)
        if np.any(adata_positions < 0):
            bad = cnmf_obs[np.where(adata_positions < 0)[0]][:10].tolist()
            raise ValueError(f"Unexpected unmatched cells after index check: {bad}")
        aligned_umap = adata_umap[adata_positions, :2]

    if aligned_umap.shape[0] != cnmf.n_obs:
        raise ValueError(
            f"Aligned UMAP has {aligned_umap.shape[0]} rows, but cNMF has {cnmf.n_obs} cells"
        )

    print(f"Transferred {obsm_key} from adata_int: {aligned_umap.shape[0]} cells x {aligned_umap.shape[1]} dimensions")
    return aligned_umap

# %%
H5MU_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/adata/cNMF_50_0_2.h5mu"
)

ADATA_INT_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/"
    "2.preprocessing_tonys_pipeline/3.integrated_days/"
    "3.all_days_concat_pc30_all_genes_sceptre_guides.h5ad"
)

if not H5MU_PATH.exists():
    raise FileNotFoundError(f"Could not find h5mu file: {H5MU_PATH}")

if not ADATA_INT_PATH.exists():
    raise FileNotFoundError(f"Could not find integrated h5ad file: {ADATA_INT_PATH}")

print("Loading h5mu...")
mdata = mu.read_h5mu(H5MU_PATH)
cnmf = mdata.mod["cNMF"]
print(f"Loaded cNMF modality: {cnmf.n_obs} cells x {cnmf.n_vars} programs")

print("Loading integrated AnnData for UMAP...")
# backed="r" avoids loading the full expression matrix; obs/obsm are enough here.
adata_int = sc.read_h5ad(ADATA_INT_PATH, backed="r")
print(f"Loaded integrated AnnData: {adata_int.n_obs} cells x {adata_int.n_vars} genes")

umap_coords = align_umap_to_cnmf_cells(
    cnmf=cnmf,
    adata_int=adata_int,
    obsm_key="X_umap",
    require_exact_match=True,
)

# Store the transferred coordinates in memory too, so downstream cells can inspect them.
cnmf.obsm["X_umap_from_adata_int"] = umap_coords

scores = cnmf.X.toarray() if hasattr(cnmf.X, "toarray") else np.asarray(cnmf.X)
print(f"scores shape = {scores.shape}")
print(f"umap_coords shape = {umap_coords.shape}")

# %%
prog_meta = pd.read_csv(REPO_ROOT / "data" / "program_activity_long_v4.csv").drop_duplicates(
    "program_name"
)
program_ids = prog_meta["program_name"].astype(int)

if "program_label" in prog_meta.columns:
    label_map = dict(zip(program_ids, prog_meta["program_label"]))
else:
    label_map = {int(program): f"P{int(program)}" for program in program_ids}

preferred_color_columns = [
    "distinct_color",
    "program_color",
    "color",
    "class_color",
]
color_column = next((col for col in preferred_color_columns if col in prog_meta.columns), None)

prog_color = {}
if color_column is not None:
    print(f"Using `{color_column}` from program metadata for colors")
    for program, color in zip(program_ids, prog_meta[color_column]):
        if pd.isna(color):
            continue
        color = str(color).strip()
        if not color:
            continue
        try:
            mpl.colors.to_rgba(color)
        except ValueError:
            continue
        prog_color[int(program)] = color
else:
    print(
        "No color column found in program metadata; generating fallback colors. "
        f"Available columns: {list(prog_meta.columns)}"
    )

program_order_path = REPO_ROOT / "analysis" / "regulator_dotplots" / "program_order.txt"
programs = []
for line in program_order_path.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    programs.append(int(line))

unique_programs = list(dict.fromkeys(programs))

fallback_cmap = plt.get_cmap("tab20", max(len(unique_programs), 1))
for idx, program in enumerate(unique_programs):
    if program not in prog_color:
        prog_color[program] = mpl.colors.to_hex(fallback_cmap(idx))

print(f"Loaded {cnmf.n_obs} cells, {len(unique_programs)} programs")
print(f"Color entries available for {len(prog_color)} programs")

# %%
# Assign dominant program per cell.
threshold = 0.02

sub_scores = scores[:, unique_programs]
max_prog_idx = np.argmax(sub_scores, axis=1)
max_score = np.max(sub_scores, axis=1)
dominant_program = np.array([unique_programs[idx] for idx in max_prog_idx])

n_cells = len(max_prog_idx)
cell_colors = np.array(["#e0e0e0"] * n_cells, dtype=object)

for program in unique_programs:
    mask = (max_score > threshold) & (dominant_program == program)
    cell_colors[mask] = prog_color.get(program, "#999999")

print(f"Assigned colors to {(max_score > threshold).sum()} cells above threshold {threshold}")

# %%
fig, ax = plt.subplots(figsize=(5, 5))

# Keep the dense UMAP point cloud rasterized in SVG as embedded image layers.
# This keeps the SVG much smaller while preserving editable text/legend elements.

grey_mask = cell_colors == "#e0e0e0"
ax.scatter(
    umap_coords[grey_mask, 0],
    umap_coords[grey_mask, 1],
    c="#e8e8e8",
    s=0.1,
    rasterized=True,
    linewidths=0,
    zorder=1,
)
for program in unique_programs:
    color = prog_color.get(program, "#999999")
    mask = (max_score > threshold) & (dominant_program == program)

    if mask.sum() > 0:
        ax.scatter(
            umap_coords[mask, 0],
            umap_coords[mask, 1],
            c=color,
            s=0.3,
            rasterized=True,
            linewidths=0,
            zorder=2,
            alpha=0.7,
        )
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect("equal")
for spine in ax.spines.values():
    spine.set_visible(False)

handles = []
for program in unique_programs:
    label = label_map.get(program, f"P{program}")
    short_label = label.split(" — ")[-1] if " — " in label else label
    handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=prog_color.get(program, "#999999"),
            markersize=3,
            label=f"P{program} {short_label}",
        )
    )

ax.legend(
    handles=handles,
    fontsize=3.5,
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    ncol=1,
    handletextpad=0.3,
    labelspacing=0.3,
)

plt.tight_layout()

pdf_path = OUT_DIR / "umap_all_cells_program_colors_integrated_umap.pdf"
png_path = OUT_DIR / "umap_all_cells_program_colors_integrated_umap.png"
svg_path = OUT_DIR / "umap_all_cells_program_colors_integrated_umap_compact.svg"

# PDF/PNG/SVG: keep the dense point cloud rasterized so files stay manageable.
# Because `svg.fonttype` is set to `none`, text and legend labels stay editable in Illustrator.
fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, bbox_inches="tight")
fig.savefig(svg_path, format="svg", bbox_inches="tight")

print(f"Saved: {pdf_path}")
print(f"Saved: {png_path}")
print(f"Saved compact SVG: {svg_path}")
plt.show()



In [ ]:
# %% [markdown]
# # All-cells UMAP colored by dominant selected cNMF program
#
# Improvements relative to the previous program-order UMAP:
#
# 1. high-contrast day-constrained palette
# 2. lighter, less conspicuous background
# 3. white halo under colored foreground cells
# 4. abundant programs plotted first, rare programs plotted last
# 5. larger foreground points and smaller background points
# 6. direct P## labels on the UMAP at each program's median location
#
# Background cells are rasterized to keep SVG size smaller. Foreground cells are
# vector by default so the colored program points remain Illustrator-editable.
# Set RASTERIZE_FOREGROUND = True if the SVG becomes too large.

# %%
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.lines import Line2D

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)

warnings.filterwarnings("ignore")

# %%
def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root by looking upward for expected project files."""
    start = start.resolve()
    required_paths = [
        Path("data/program_activity_long_v4.csv"),
        Path("analysis/regulator_dotplots/program_order.txt"),
    ]

    for candidate in [start, *start.parents]:
        if all((candidate / rel_path).exists() for rel_path in required_paths):
            return candidate

    raise FileNotFoundError(
        "Could not auto-detect REPO_ROOT. Set REPO_ROOT manually in this cell."
    )


REPO_ROOT = find_repo_root()
OUT_DIR = REPO_ROOT / "analysis" / "regulator_dotplots" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FONT_DIR = REPO_ROOT / ".pixi" / "envs" / "default" / "fonts"
if FONT_DIR.exists():
    for font_file in fm.findSystemFonts(fontpaths=[str(FONT_DIR)]):
        fm.fontManager.addfont(font_file)

mpl.rcParams.update(
    {
        "font.family": "Arial",
        "font.size": 7,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",  # keep text editable in Illustrator
    }
)

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"OUT_DIR   = {OUT_DIR}")

# %%
# -----------------------------
# Input paths
# -----------------------------
H5MU_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/adata/cNMF_50_0_2.h5mu"
)

ADATA_INT_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/"
    "2.preprocessing_tonys_pipeline/3.integrated_days/"
    "3.all_days_concat_pc30_all_genes_sceptre_guides.h5ad"
)

PROGRAM_META_PATH = REPO_ROOT / "data" / "program_activity_long_v4.csv"
PROGRAM_ORDER_PATH = REPO_ROOT / "analysis" / "regulator_dotplots" / "program_order.txt"

# %%
# -----------------------------
# Plot settings
# -----------------------------
THRESHOLD = 0.02

# "global": color a cell only if its dominant program across ALL cNMF programs
#           is listed in program_order.txt.
# "program_order": old behavior; color by highest-scoring program among only
#                  the programs listed in program_order.txt.
DOMINANCE_MODE = "global"

# Lighter and less conspicuous than the previous gray/lavender background.
# This also avoids competing with D0 gray programs.
BACKGROUND_COLOR = "#F2F5FA"
BACKGROUND_ALPHA = 0.42
BACKGROUND_SIZE = 0.10

# Larger foreground points. Foreground is vector by default for Illustrator editing.
FOREGROUND_SIZE = 0.85
FOREGROUND_ALPHA = 0.95
RASTERIZE_FOREGROUND = False

# White halo under colored cells to separate similar within-day shades.
USE_WHITE_HALO = True
HALO_COLOR = "white"
HALO_ALPHA = 0.80
HALO_SIZE_MULTIPLIER = 2.6
RASTERIZE_HALO = RASTERIZE_FOREGROUND

# Direct labels near each program region.
ANNOTATE_PROGRAMS = False
LABEL_FONT_SIZE = 6.0
LABEL_ALPHA = 0.92
LABEL_BOX_ALPHA = 0.78

# Optional manual label offsets in UMAP units, if two labels overlap.
# Example: LABEL_OFFSETS = {23: (0.25, -0.1), 15: (-0.15, 0.2)}
LABEL_OFFSETS = {}

FIGSIZE = (5.2, 5.2)
OUTPUT_STEM = OUT_DIR / "umap_program_order_high_contrast_halo_labels"

# %%
# High-contrast palette constrained by day/color-family rules.
# D0: gray; D1: orange; D2: darker orange/brown;
# D3 arterial EC: red; D3 non-EC: blue.
HIGH_CONTRAST_PROGRAM_COLORS = {
    # Day 0 programs: grays
    22: "#1F1F1F",
    29: "#5C5C5C",
    7:  "#9E9E9E",
    49: "#C7C7C7",

    # Day 1 programs: oranges
    30: "#FFD166",
    28: "#FFA600",
    13: "#F97316",
    48: "#C2410C",

    # Day 2 programs: darker oranges/browns
    15: "#B45309",
    23: "#92400E",
    19: "#7C2D12",
    17: "#5F2608",
    14: "#3F1D05",

    # Day 3 arterial EC programs: reds
    11: "#FB6A4A",
    2:  "#A50F15",

    # Day 3 non-EC programs: blues
    34: "#B3E5FC",
    27: "#56B4E9",
    24: "#1F78B4",
    44: "#003F7F",
    33: "#001D4A",
}

# Optional family labels for summary output.
PROGRAM_COLOR_FAMILY = {
    22: "D0_gray", 29: "D0_gray", 7: "D0_gray", 49: "D0_gray",
    30: "D1_orange", 28: "D1_orange", 13: "D1_orange", 48: "D1_orange",
    15: "D2_dark_orange", 23: "D2_dark_orange", 19: "D2_dark_orange", 17: "D2_dark_orange", 14: "D2_dark_orange",
    11: "D3_arterial_red", 2: "D3_arterial_red",
    34: "D3_nonEC_blue", 27: "D3_nonEC_blue", 24: "D3_nonEC_blue", 44: "D3_nonEC_blue", 33: "D3_nonEC_blue",
}

# %%
def align_umap_to_cnmf_cells(
    *,
    cnmf,
    adata_int,
    obsm_key: str = "X_umap",
    require_exact_match: bool = True,
) -> np.ndarray:
    """Return adata_int.obsm[obsm_key] reordered to cnmf.obs_names."""
    cnmf_obs = pd.Index(cnmf.obs_names, name="cnmf_obs_names")
    adata_obs = pd.Index(adata_int.obs_names, name="adata_int_obs_names")

    if not cnmf_obs.is_unique:
        duplicated = cnmf_obs[cnmf_obs.duplicated()].unique()[:10].tolist()
        raise ValueError(f"cnmf.obs_names contains duplicate cell IDs, e.g. {duplicated}")

    if not adata_obs.is_unique:
        duplicated = adata_obs[adata_obs.duplicated()].unique()[:10].tolist()
        raise ValueError(f"adata_int.obs_names contains duplicate cell IDs, e.g. {duplicated}")

    if obsm_key not in adata_int.obsm_keys():
        raise KeyError(
            f"{obsm_key!r} was not found in adata_int.obsm. "
            f"Available obsm keys: {list(adata_int.obsm_keys())}"
        )

    missing_in_adata = cnmf_obs[~cnmf_obs.isin(adata_obs)]
    extra_in_adata = adata_obs[~adata_obs.isin(cnmf_obs)]

    if len(missing_in_adata) > 0:
        raise ValueError(
            "Some cNMF cells are missing from adata_int. "
            f"n_missing={len(missing_in_adata)}; first few: {missing_in_adata[:10].tolist()}"
        )

    if require_exact_match and len(extra_in_adata) > 0:
        raise ValueError(
            "adata_int contains cells that are not present in cNMF. "
            f"n_extra={len(extra_in_adata)}; first few: {extra_in_adata[:10].tolist()}"
        )

    adata_umap = np.asarray(adata_int.obsm[obsm_key])
    if adata_umap.ndim != 2 or adata_umap.shape[1] < 2:
        raise ValueError(
            f"adata_int.obsm[{obsm_key!r}] should be a 2D array with at least 2 columns; "
            f"got shape {adata_umap.shape}"
        )

    if cnmf_obs.equals(adata_obs):
        print("Cell indexes match in the same order; using adata_int X_umap directly.")
        aligned_umap = adata_umap[:, :2]
    else:
        print("Cell indexes match as IDs but not in the same order; reordering adata_int X_umap to cnmf.obs_names.")
        adata_positions = adata_obs.get_indexer(cnmf_obs)
        if np.any(adata_positions < 0):
            bad = cnmf_obs[np.where(adata_positions < 0)[0]][:10].tolist()
            raise ValueError(f"Unexpected unmatched cells after index check: {bad}")
        aligned_umap = adata_umap[adata_positions, :2]

    print(f"Transferred {obsm_key} from adata_int: {aligned_umap.shape[0]} cells x {aligned_umap.shape[1]} dimensions")
    return aligned_umap


def parse_program_id(x):
    """Parse cNMF program IDs from var_names like '13' or 'P13'."""
    s = str(x)
    if s.startswith("P"):
        s = s[1:]
    return int(s)

# %%
for path in [H5MU_PATH, ADATA_INT_PATH, PROGRAM_META_PATH, PROGRAM_ORDER_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Loading h5mu...")
mdata = mu.read_h5mu(H5MU_PATH)
cnmf = mdata.mod["cNMF"]
print(f"Loaded cNMF modality: {cnmf.n_obs} cells x {cnmf.n_vars} programs")

print("Loading integrated AnnData for UMAP...")
adata_int = sc.read_h5ad(ADATA_INT_PATH, backed="r")
print(f"Loaded integrated AnnData: {adata_int.n_obs} cells x {adata_int.n_vars} genes")

umap_coords = align_umap_to_cnmf_cells(
    cnmf=cnmf,
    adata_int=adata_int,
    obsm_key="X_umap",
    require_exact_match=True,
)

scores = cnmf.X.toarray() if hasattr(cnmf.X, "toarray") else np.asarray(cnmf.X)
print(f"scores shape = {scores.shape}")
print(f"umap_coords shape = {umap_coords.shape}")

# %%
# Load metadata labels and default colors, then override selected program colors
# with the high-contrast day-constrained palette above.
prog_meta = pd.read_csv(PROGRAM_META_PATH).drop_duplicates("program_name")
prog_meta["program_name"] = prog_meta["program_name"].astype(int)
program_ids = prog_meta["program_name"].astype(int)

if "program_label" in prog_meta.columns:
    label_map = dict(zip(program_ids, prog_meta["program_label"].astype(str)))
else:
    label_map = {int(program): f"P{int(program)}" for program in program_ids}

preferred_color_columns = ["distinct_color", "program_color", "color", "class_color"]
color_column = next((col for col in preferred_color_columns if col in prog_meta.columns), None)

prog_color = {}
if color_column is not None:
    print(f"Using `{color_column}` from program metadata as baseline colors")
    for program, color in zip(program_ids, prog_meta[color_column]):
        if pd.isna(color):
            continue
        color = str(color).strip()
        if not color:
            continue
        try:
            mpl.colors.to_rgba(color)
        except ValueError:
            continue
        prog_color[int(program)] = color

# Override with the high-contrast colors for the selected program-order programs.
prog_color.update(HIGH_CONTRAST_PROGRAM_COLORS)

# Parse program_order.txt.
programs = []
for line in PROGRAM_ORDER_PATH.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    programs.append(int(line))

unique_programs = list(dict.fromkeys(programs))
print(f"Loaded {len(unique_programs)} unique programs from program_order.txt:")
print(unique_programs)

# Map cNMF score columns to zero-based program IDs.
try:
    cnmf_program_ids = [parse_program_id(v) for v in cnmf.var_names]
except Exception:
    cnmf_program_ids = list(range(scores.shape[1]))

if len(cnmf_program_ids) != scores.shape[1]:
    raise ValueError("Could not map cNMF score columns to program IDs.")

program_to_col = {pid: idx for idx, pid in enumerate(cnmf_program_ids)}
missing_from_scores = [p for p in unique_programs if p not in program_to_col]
if missing_from_scores:
    raise ValueError(f"These program_order programs are missing from cNMF score matrix: {missing_from_scores}")

# Fallback colors for any program in program_order not explicitly assigned.
fallback_cmap = plt.get_cmap("tab20", max(len(unique_programs), 1))
for idx, program in enumerate(unique_programs):
    if program not in prog_color:
        prog_color[program] = mpl.colors.to_hex(fallback_cmap(idx))

print(f"Color entries available for {len(prog_color)} programs")

# Save the exact color table used in this plot.
color_table = pd.DataFrame(
    {
        "program": unique_programs,
        "program_label": [label_map.get(p, f"P{p}") for p in unique_programs],
        "hex_color": [prog_color[p] for p in unique_programs],
        "color_family": [PROGRAM_COLOR_FAMILY.get(p, "fallback_or_metadata") for p in unique_programs],
    }
)
color_table.to_csv(OUTPUT_STEM.with_suffix(".colors_used.csv"), index=False)
display(color_table)

# %%
# Assign dominant program per cell.
if DOMINANCE_MODE == "global":
    dominant_col_idx = np.argmax(scores, axis=1)
    max_score = scores[np.arange(scores.shape[0]), dominant_col_idx]
    dominant_program = np.array([cnmf_program_ids[idx] for idx in dominant_col_idx], dtype=int)
    selected_mask = np.isin(dominant_program, unique_programs) & (max_score > THRESHOLD)
elif DOMINANCE_MODE == "program_order":
    selected_cols = [program_to_col[p] for p in unique_programs]
    sub_scores = scores[:, selected_cols]
    max_prog_idx = np.argmax(sub_scores, axis=1)
    max_score = np.max(sub_scores, axis=1)
    dominant_program = np.array([unique_programs[idx] for idx in max_prog_idx], dtype=int)
    selected_mask = max_score > THRESHOLD
else:
    raise ValueError("DOMINANCE_MODE must be 'global' or 'program_order'")

summary_records = []
for program in unique_programs:
    mask = selected_mask & (dominant_program == program)
    summary_records.append(
        {
            "program": program,
            "program_label": label_map.get(program, f"P{program}"),
            "hex_color": prog_color.get(program, "#999999"),
            "color_family": PROGRAM_COLOR_FAMILY.get(program, "fallback_or_metadata"),
            "n_cells": int(mask.sum()),
        }
    )
summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(OUTPUT_STEM.with_suffix(".summary.csv"), index=False)
display(summary_df)

print(f"DOMINANCE_MODE = {DOMINANCE_MODE!r}")
print(f"Assigned colors to {int(selected_mask.sum()):,} cells above threshold {THRESHOLD}")
print(f"Background cells: {int((~selected_mask).sum()):,}")

# %%
# Plot abundant programs first and rare programs last so rare programs stay visible.
program_counts = {
    program: int((selected_mask & (dominant_program == program)).sum())
    for program in unique_programs
}
plot_order = sorted(unique_programs, key=lambda p: program_counts[p], reverse=True)
legend_order = unique_programs  # keep legend in program_order.txt order

fig, ax = plt.subplots(figsize=FIGSIZE)

x = umap_coords[:, 0]
y = umap_coords[:, 1]

# Background cells: rasterized, small, light.
bg_mask = ~selected_mask
ax.scatter(
    x[bg_mask],
    y[bg_mask],
    c=BACKGROUND_COLOR,
    s=BACKGROUND_SIZE,
    alpha=BACKGROUND_ALPHA,
    rasterized=True,
    linewidths=0,
    edgecolors="none",
    zorder=1,
)

# Foreground cells: halo + color.
for program in plot_order:
    mask = selected_mask & (dominant_program == program)
    if mask.sum() == 0:
        continue

    if USE_WHITE_HALO:
        ax.scatter(
            x[mask],
            y[mask],
            c=HALO_COLOR,
            s=FOREGROUND_SIZE * HALO_SIZE_MULTIPLIER,
            alpha=HALO_ALPHA,
            rasterized=RASTERIZE_HALO,
            linewidths=0,
            edgecolors="none",
            zorder=2,
        )

    ax.scatter(
        x[mask],
        y[mask],
        c=prog_color.get(program, "#999999"),
        s=FOREGROUND_SIZE,
        alpha=FOREGROUND_ALPHA,
        rasterized=RASTERIZE_FOREGROUND,
        linewidths=0,
        edgecolors="none",
        zorder=3,
    )

# Direct P labels near robust median position of each program.
if ANNOTATE_PROGRAMS:
    for program in legend_order:
        mask = selected_mask & (dominant_program == program)
        if mask.sum() == 0:
            continue
        x_med = float(np.median(x[mask]))
        y_med = float(np.median(y[mask]))
        dx, dy = LABEL_OFFSETS.get(program, (0.0, 0.0))
        ax.text(
            x_med + dx,
            y_med + dy,
            f"P{program}",
            color=prog_color.get(program, "#999999"),
            fontsize=LABEL_FONT_SIZE,
            fontweight="bold",
            ha="center",
            va="center",
            alpha=LABEL_ALPHA,
            bbox=dict(
                boxstyle="round,pad=0.18",
                facecolor="white",
                edgecolor="none",
                alpha=LABEL_BOX_ALPHA,
            ),
            zorder=10,
        )

ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect("equal")
for spine in ax.spines.values():
    spine.set_visible(False)

handles = []
for program in legend_order:
    label = label_map.get(program, f"P{program}")
    short_label = label.split(" — ")[-1] if " — " in label else label
    handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=prog_color.get(program, "#999999"),
            markeredgecolor="white",
            markeredgewidth=0.8,
            markersize=4.2,
            label=f"P{program} {short_label}",
        )
    )

ax.legend(
    handles=handles,
    fontsize=4.2,
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    ncol=1,
    handletextpad=0.35,
    labelspacing=0.35,
)

plt.tight_layout()

pdf_path = OUTPUT_STEM.with_suffix(".pdf")
png_path = OUTPUT_STEM.with_suffix(".png")
svg_path = OUTPUT_STEM.with_suffix(".svg")

fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, bbox_inches="tight")
fig.savefig(svg_path, format="svg", bbox_inches="tight")

print(f"Saved: {pdf_path}")
print(f"Saved: {png_path}")
print(f"Saved editable SVG: {svg_path}")
print(f"Saved color table: {OUTPUT_STEM.with_suffix('.colors_used.csv')}")
print(f"Saved summary table: {OUTPUT_STEM.with_suffix('.summary.csv')}")
plt.show()


#### UMAP for program 13, 30, 15, 17

In [ ]:
# %% [markdown]
# # One UMAP for selected programs with rasterized background
#
# This script makes a single UMAP colored only for cells whose **globally dominant**
# cNMF program is one of:
# - P13
# - P30
# - P15
# - P17
#
# All other cells are shown as background.
#
# Background points are rasterized to keep the SVG file size small.
# Foreground highlighted points remain vector, and text stays editable in Illustrator.

# %%
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# %%
def find_repo_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()
    required_paths = [Path("data/program_activity_long_v4.csv")]
    for candidate in [start, *start.parents]:
        if all((candidate / rel_path).exists() for rel_path in required_paths):
            return candidate
    raise FileNotFoundError("Could not auto-detect REPO_ROOT. Set REPO_ROOT manually.")

REPO_ROOT = find_repo_root()
OUT_DIR = REPO_ROOT / "analysis" / "regulator_dotplots" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FONT_DIR = REPO_ROOT / ".pixi" / "envs" / "default" / "fonts"
if FONT_DIR.exists():
    for font_file in fm.findSystemFonts(fontpaths=[str(FONT_DIR)]):
        fm.fontManager.addfont(font_file)

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"OUT_DIR   = {OUT_DIR}")

# %%
def align_umap_to_cnmf_cells(*, cnmf, adata_int, obsm_key="X_umap", require_exact_match=True):
    cnmf_obs = pd.Index(cnmf.obs_names)
    adata_obs = pd.Index(adata_int.obs_names)

    if obsm_key not in adata_int.obsm_keys():
        raise KeyError(f"{obsm_key!r} not found in adata_int.obsm")

    missing_in_adata = cnmf_obs[~cnmf_obs.isin(adata_obs)]
    extra_in_adata = adata_obs[~adata_obs.isin(cnmf_obs)]
    if len(missing_in_adata) > 0:
        raise ValueError(f"Some cNMF cells are missing from adata_int: {missing_in_adata[:10].tolist()}")
    if require_exact_match and len(extra_in_adata) > 0:
        raise ValueError(f"adata_int contains extra cells not in cNMF: {extra_in_adata[:10].tolist()}")

    adata_umap = np.asarray(adata_int.obsm[obsm_key])
    if cnmf_obs.equals(adata_obs):
        aligned_umap = adata_umap[:, :2]
    else:
        positions = adata_obs.get_indexer(cnmf_obs)
        aligned_umap = adata_umap[positions, :2]

    print(f"Transferred {obsm_key}: {aligned_umap.shape[0]} cells x {aligned_umap.shape[1]} dims")
    return aligned_umap


def parse_program_id(x):
    s = str(x)
    return int(s[1:]) if s.startswith("P") else int(s)

# %%
H5MU_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/adata/cNMF_50_0_2.h5mu"
)
ADATA_INT_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/"
    "2.preprocessing_tonys_pipeline/3.integrated_days/"
    "3.all_days_concat_pc30_all_genes_sceptre_guides.h5ad"
)
PROGRAM_META_PATH = REPO_ROOT / "data" / "program_activity_long_v4.csv"
# PROGRAM_META_PATH = Path("/mnt/data/program_activity_long_v4_updated_colors_labels.csv")

SELECTED_PROGRAMS = [13, 30, 15, 17]
THRESHOLD = 0.02

BACKGROUND_COLOR = "#E6ECF5"
BACKGROUND_ALPHA = 0.28
BACKGROUND_SIZE = 0.22
FOREGROUND_ALPHA = 0.95
FOREGROUND_SIZE = 0.7
FOREGROUND_EDGEWIDTH = 0.0
FIG_W = 4.2
FIG_H = 4.2
OUTPUT_STEM = OUT_DIR / "umap_P13_P30_P15_P17_one_umap_global_dominant_rasterized_background"

# %%
print("Loading h5mu...")
mdata = mu.read_h5mu(H5MU_PATH)
cnmf = mdata.mod["cNMF"]
print(f"Loaded cNMF modality: {cnmf.n_obs} cells x {cnmf.n_vars} programs")

print("Loading integrated AnnData for UMAP...")
adata_int = sc.read_h5ad(ADATA_INT_PATH, backed="r")
print(f"Loaded integrated AnnData: {adata_int.n_obs} cells x {adata_int.n_vars} genes")

umap_coords = align_umap_to_cnmf_cells(cnmf=cnmf, adata_int=adata_int, obsm_key="X_umap", require_exact_match=True)
scores = cnmf.X.toarray() if hasattr(cnmf.X, "toarray") else np.asarray(cnmf.X)
print(f"scores shape = {scores.shape}")
print(f"umap_coords shape = {umap_coords.shape}")

# %%
prog_meta = pd.read_csv(PROGRAM_META_PATH).drop_duplicates("program_name")
prog_meta["program_name"] = prog_meta["program_name"].astype(int)
label_map = dict(zip(prog_meta["program_name"], prog_meta.get("program_label", prog_meta["program_name"].astype(str)).astype(str)))

preferred_color_columns = ["distinct_color", "program_color", "color", "class_color"]
color_column = next((c for c in preferred_color_columns if c in prog_meta.columns), None)
if color_column is None:
    raise ValueError(f"No usable color column found. Available columns: {list(prog_meta.columns)}")
prog_color = {}
for program, color in zip(prog_meta["program_name"], prog_meta[color_column]):
    if pd.isna(color):
        continue
    color = str(color).strip()
    try:
        mpl.colors.to_rgba(color)
        prog_color[int(program)] = color
    except Exception:
        pass

cnmf_program_ids = [parse_program_id(v) for v in cnmf.var_names] if len(cnmf.var_names) == scores.shape[1] else list(range(scores.shape[1]))
program_to_col = {pid: idx for idx, pid in enumerate(cnmf_program_ids)}

for p in SELECTED_PROGRAMS:
    if p not in program_to_col:
        raise ValueError(f"Selected program P{p} not found among cNMF columns")
    if p not in prog_color:
        raise ValueError(f"Selected program P{p} missing a valid color in metadata")
    label_map.setdefault(p, f"P{p}")

print("Selected programs:")
for p in SELECTED_PROGRAMS:
    print(f"  P{p}: column {program_to_col[p]}, label={label_map[p]}, color={prog_color[p]}")

# %%
# Global dominant program across all cNMF programs.
dominant_col_idx = np.argmax(scores, axis=1)
dominant_score = scores[np.arange(scores.shape[0]), dominant_col_idx]
dominant_program_all = np.array([cnmf_program_ids[idx] for idx in dominant_col_idx], dtype=int)

# Only color cells whose globally dominant program is one of the selected programs.
selected_mask = np.isin(dominant_program_all, SELECTED_PROGRAMS) & (dominant_score > THRESHOLD)

summary_records = []
for p in SELECTED_PROGRAMS:
    mask = selected_mask & (dominant_program_all == p)
    summary_records.append({
        "program": p,
        "program_label": label_map[p],
        "hex_color": prog_color[p],
        "n_cells": int(mask.sum()),
    })
summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(OUTPUT_STEM.with_suffix('.summary.csv'), index=False)
print(summary_df)
print("selected cells:", int(selected_mask.sum()))
print("background cells:", int((~selected_mask).sum()))

# %%
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), constrained_layout=True)
x = umap_coords[:, 0]
y = umap_coords[:, 1]

# Background first, rasterized.
bg_mask = ~selected_mask
ax.scatter(
    x[bg_mask], y[bg_mask],
    s=BACKGROUND_SIZE, c=BACKGROUND_COLOR, alpha=BACKGROUND_ALPHA,
    linewidths=0, edgecolors="none", rasterized=True, zorder=1,
)

# Foreground selected programs, vector.
for p in SELECTED_PROGRAMS:
    mask = selected_mask & (dominant_program_all == p)
    ax.scatter(
        x[mask], y[mask],
        s=FOREGROUND_SIZE, c=prog_color[p], alpha=FOREGROUND_ALPHA,
        linewidths=FOREGROUND_EDGEWIDTH, edgecolors="none", rasterized=False,
        zorder=2, label=f"P{p}: {label_map[p]} (n={int(mask.sum()):,})",
    )

ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect("equal")
for spine in ax.spines.values():
    spine.set_visible(False)

legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=4.5,
           markerfacecolor=BACKGROUND_COLOR, markeredgewidth=0, alpha=0.75,
           label=f"Background (n={int(bg_mask.sum()):,})")
]
for p in SELECTED_PROGRAMS:
    mask = selected_mask & (dominant_program_all == p)
    legend_elements.append(
        Line2D([0], [0], marker='o', linestyle='', markersize=4.5,
               markerfacecolor=prog_color[p], markeredgewidth=0,
               label=f"P{p}: {label_map[p]} (n={int(mask.sum()):,})")
    )

ax.legend(handles=legend_elements, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=7, handletextpad=0.5, borderaxespad=0.0)
ax.set_title(
    f"Integrated UMAP colored by globally dominant selected program\n"
    f"Only cells whose global dominant program is in {', '.join([f'P{p}' for p in SELECTED_PROGRAMS])} are colored; threshold > {THRESHOLD}",
    fontsize=8, pad=6,
)

for ext in ["svg", "pdf", "png"]:
    fig.savefig(OUTPUT_STEM.with_suffix(f".{ext}"), dpi=300 if ext == "png" else None)

plt.show()
plt.close(fig)

print("Saved outputs:")
print(OUTPUT_STEM.with_suffix('.svg'))
print(OUTPUT_STEM.with_suffix('.pdf'))
print(OUTPUT_STEM.with_suffix('.png'))
print(OUTPUT_STEM.with_suffix('.summary.csv'))


In [ ]:
# %% [markdown]
# # Display programs from the metadata CSV one by one on UMAP
#
# This notebook plots each program listed in `data/program_activity_long_v4.csv` on the same UMAP.
#
# Default behavior:
# - uses cNMF scores from the `.h5mu` file;
# - uses `X_umap` transferred from the integrated `.h5ad` file, aligned by cell index;
# - displays every program one by one;
# - saves compact SVG/PNG files for each program.

# %%
import re
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# %%
def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root by looking upward for the program metadata CSV."""
    start = start.resolve()
    required_paths = [Path("analysis/regulator_dotplots/program_order.txt")]

    for candidate in [start, *start.parents]:
        if all((candidate / rel_path).exists() for rel_path in required_paths):
            return candidate

    raise FileNotFoundError(
        "Could not auto-detect REPO_ROOT. Set REPO_ROOT manually in this cell."
    )


# In a notebook, __file__ is usually not defined, so use an auto-detected repo root.
# If auto-detection fails, replace this with: REPO_ROOT = Path("/path/to/your/repo")
REPO_ROOT = find_repo_root()

OUT_DIR = REPO_ROOT / "analysis" / "regulator_dotplots" / "figures" / "umap_programs_one_by_one"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FONT_DIR = REPO_ROOT / ".pixi" / "envs" / "default" / "fonts"
if FONT_DIR.exists():
    for font_file in fm.findSystemFonts(fontpaths=[str(FONT_DIR)]):
        fm.fontManager.addfont(font_file)

mpl.rcParams.update(
    {
        "font.family": "Arial",
        "font.size": 7,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        # Keep labels/text editable in Illustrator while rasterizing dense point layers.
        "svg.fonttype": "none",
    }
)

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"OUT_DIR   = {OUT_DIR}")

# %%
H5MU_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/"
    "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/adata/cNMF_50_0_2.h5mu"
)

# Set to False to use cnmf.obsm["X_umap"] instead of the integrated h5ad UMAP.
USE_ADATA_INT_UMAP = True
ADATA_INT_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/"
    "2.preprocessing_tonys_pipeline/3.integrated_days/"
    "3.all_days_concat_pc30_all_genes_sceptre_guides.h5ad"
)

# If True, error when adata_int has cells not present in cnmf or vice versa.
# If False, adata_int may contain extra cells, but every cnmf cell must still be present.
REQUIRE_EXACT_OBS_MATCH = True

PROGRAM_CSV_PATH = REPO_ROOT / "data" / "program_activity_long_v4.csv"

# %%
def dense_array(x):
    """Convert sparse/dask-like arrays to a NumPy array when needed."""
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)


def transfer_umap_from_adata_int(cnmf, adata_int, *, require_exact_match: bool = True) -> np.ndarray:
    """Return adata_int X_umap reordered to match cnmf.obs_names."""
    if "X_umap" not in adata_int.obsm:
        raise KeyError("adata_int.obsm does not contain 'X_umap'.")

    cnmf_index = pd.Index(cnmf.obs_names, name="cnmf_obs_names")
    adata_index = pd.Index(adata_int.obs_names, name="adata_int_obs_names")

    if not cnmf_index.is_unique:
        duplicated = cnmf_index[cnmf_index.duplicated()].unique()[:5].tolist()
        raise ValueError(f"cnmf.obs_names is not unique. Examples: {duplicated}")
    if not adata_index.is_unique:
        duplicated = adata_index[adata_index.duplicated()].unique()[:5].tolist()
        raise ValueError(f"adata_int.obs_names is not unique. Examples: {duplicated}")

    missing_in_adata = cnmf_index.difference(adata_index)
    extra_in_adata = adata_index.difference(cnmf_index)

    if len(missing_in_adata) > 0:
        examples = missing_in_adata[:10].tolist()
        raise ValueError(
            f"{len(missing_in_adata):,} cnmf cells are missing from adata_int. "
            f"Examples: {examples}"
        )

    if require_exact_match and len(extra_in_adata) > 0:
        examples = extra_in_adata[:10].tolist()
        raise ValueError(
            f"adata_int has {len(extra_in_adata):,} extra cells not present in cnmf. "
            f"Examples: {examples}. If this is expected, set REQUIRE_EXACT_OBS_MATCH = False."
        )

    positions = adata_index.get_indexer(cnmf_index)
    if np.any(positions < 0):
        raise RuntimeError("Unexpected failed indexer values after missing-cell check.")

    adata_umap = np.asarray(adata_int.obsm["X_umap"])
    umap_coords = adata_umap[positions, :]

    if umap_coords.shape[0] != cnmf.n_obs or umap_coords.shape[1] != 2:
        raise ValueError(
            "Transferred UMAP has unexpected shape: "
            f"{umap_coords.shape}; expected ({cnmf.n_obs}, 2)."
        )

    print("Transferred X_umap from adata_int to cnmf order")
    print(f"  cnmf cells:      {len(cnmf_index):,}")
    print(f"  adata_int cells: {len(adata_index):,}")
    print(f"  extra adata_int cells ignored: {len(extra_in_adata):,}")
    return umap_coords


if not H5MU_PATH.exists():
    raise FileNotFoundError(f"Could not find h5mu file: {H5MU_PATH}")

print("Loading h5mu...")
mdata = mu.read_h5mu(H5MU_PATH)
cnmf = mdata.mod["cNMF"]
scores = dense_array(cnmf.X)

if USE_ADATA_INT_UMAP:
    if not ADATA_INT_PATH.exists():
        raise FileNotFoundError(f"Could not find integrated h5ad file: {ADATA_INT_PATH}")
    import scanpy as sc

    print("Loading integrated h5ad for UMAP transfer...")
    adata_int = sc.read_h5ad(ADATA_INT_PATH, backed="r")
    umap_coords = transfer_umap_from_adata_int(
        cnmf,
        adata_int,
        require_exact_match=REQUIRE_EXACT_OBS_MATCH,
    )
    cnmf.obsm["X_umap_from_adata_int"] = umap_coords
    try:
        adata_int.file.close()
    except Exception:
        pass
else:
    if "X_umap" not in cnmf.obsm:
        raise KeyError("cnmf.obsm does not contain 'X_umap'.")
    umap_coords = np.asarray(cnmf.obsm["X_umap"])

print(f"Loaded cNMF modality: {cnmf.n_obs:,} cells x {cnmf.n_vars:,} programs")
print(f"UMAP coordinates shape: {umap_coords.shape}")
print(f"Scores shape: {scores.shape}")

# %%
def coerce_program_id(value):
    """Convert program_name values such as 3, 3.0, or 'P3' to integer IDs."""
    if pd.isna(value):
        return None
    try:
        return int(value)
    except Exception:
        match = re.search(r"\d+", str(value))
        return int(match.group(0)) if match else None


def is_valid_color(value) -> bool:
    if pd.isna(value):
        return False
    try:
        mpl.colors.to_hex(value)
        return True
    except ValueError:
        return False


def safe_filename(value: str, max_len: int = 90) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")
    return cleaned[:max_len] if cleaned else "program"


def short_label(label: str) -> str:
    label = str(label)
    return label.split(" — ")[-1] if " — " in label else label


prog_meta_raw = pd.read_csv(PROGRAM_CSV_PATH)
if "program_name" not in prog_meta_raw.columns:
    raise KeyError(
        f"Expected a 'program_name' column in {PROGRAM_CSV_PATH}. "
        f"Available columns: {list(prog_meta_raw.columns)}"
    )

prog_meta = prog_meta_raw.copy()
prog_meta["program_id"] = prog_meta["program_name"].map(coerce_program_id)
prog_meta = prog_meta.dropna(subset=["program_id"]).copy()
prog_meta["program_id"] = prog_meta["program_id"].astype(int)
prog_meta = prog_meta.drop_duplicates("program_id", keep="first").reset_index(drop=True)

programs_from_csv = prog_meta["program_id"].tolist()

if "program_label" in prog_meta.columns:
    label_map = dict(zip(prog_meta["program_id"], prog_meta["program_label"].astype(str)))
else:
    label_map = {program: f"P{program}" for program in programs_from_csv}

preferred_color_columns = ["distinct_color", "program_color", "color", "class_color"]
color_column = next((col for col in preferred_color_columns if col in prog_meta.columns), None)

prog_color = {}
if color_column is not None:
    for program, color in zip(prog_meta["program_id"], prog_meta[color_column]):
        if is_valid_color(color):
            prog_color[int(program)] = mpl.colors.to_hex(color)
    print(f"Using '{color_column}' from program metadata for program colors")
else:
    print("No color column found in program metadata; using fallback tab20 colors")

fallback_cmap = plt.get_cmap("tab20")
for idx, program in enumerate(programs_from_csv):
    if program not in prog_color:
        prog_color[program] = mpl.colors.to_hex(fallback_cmap(idx % fallback_cmap.N))

print(f"Loaded {len(programs_from_csv):,} unique programs from {PROGRAM_CSV_PATH}")
print(f"Program metadata columns: {list(prog_meta_raw.columns)}")

# %%
def resolve_program_columns(program_ids, cnmf, scores) -> tuple[dict[int, int], list[int]]:
    """Map program IDs from the CSV to columns in cnmf.X."""
    n_program_cols = scores.shape[1]
    var_names = [str(name) for name in getattr(cnmf, "var_names", [])]
    var_lookup = {name: idx for idx, name in enumerate(var_names)}

    program_to_col = {}
    skipped = []

    for program in program_ids:
        candidates = [
            str(program),
            f"P{program}",
            f"p{program}",
            f"program_{program}",
            f"Program_{program}",
            f"cNMF_{program}",
        ]

        matched_col = None
        for candidate in candidates:
            if candidate in var_lookup:
                matched_col = var_lookup[candidate]
                break

        # Fallback: previous scripts used program IDs directly as integer column indices.
        if matched_col is None and 0 <= int(program) < n_program_cols:
            matched_col = int(program)

        if matched_col is None:
            skipped.append(int(program))
        else:
            program_to_col[int(program)] = int(matched_col)

    return program_to_col, skipped


program_to_col, skipped_programs = resolve_program_columns(programs_from_csv, cnmf, scores)
programs = [program for program in programs_from_csv if program in program_to_col]

if len(programs) == 0:
    raise ValueError(
        "None of the programs from the CSV could be mapped to columns in cnmf.X. "
        "Check whether program_name is 0-based, 1-based, or stored in cnmf.var_names."
    )

if skipped_programs:
    print(
        f"Skipped {len(skipped_programs):,} programs that could not be mapped to cnmf.X columns: "
        f"{skipped_programs[:20]}"
    )

print(f"Will plot {len(programs):,} programs from the CSV")
print("First mapped programs:")
for program in programs[:10]:
    print(f"  P{program} -> scores[:, {program_to_col[program]}] | {label_map.get(program, f'P{program}')}")

# %%
# Plot controls.
PLOT_MODE = "score"  # "score" or "dominant"

# Score mode: plot the activity score of one program across cells.
SCORE_MIN = 0.0
SCORE_VMAX_QUANTILE = 0.99

# Dominant mode: highlight cells for which the current program is the highest-scoring program.
DOMINANT_THRESHOLD = 0.02

FIGSIZE = (4.2, 4.2)
BACKGROUND_SIZE = 0.12
POINT_SIZE = 0.35
BACKGROUND_COLOR = "#e8e8e8"
HIGHLIGHT_ALPHA = 0.85

SAVE_FORMATS = ("svg", "png")

# Precompute dominant program among the CSV-listed programs for optional dominant mode.
program_cols = [program_to_col[program] for program in programs]
sub_scores = scores[:, program_cols]
max_program_idx = np.argmax(sub_scores, axis=1)
max_score = np.max(sub_scores, axis=1)
dominant_program = np.array([programs[idx] for idx in max_program_idx])

print("Plot controls loaded")

# %%
def make_program_cmap(program: int):
    """Make a light-to-program-color colormap for score plots."""
    color = prog_color.get(program, "#377eb8")
    if not is_valid_color(color):
        color = "#377eb8"
    return mpl.colors.LinearSegmentedColormap.from_list(
        f"program_{program}_cmap",
        ["#f0f0f0", mpl.colors.to_hex(color)],
    )


def style_umap_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_visible(False)


def plot_program(
    program: int,
    *,
    mode: str = PLOT_MODE,
    save: bool = False,
    show: bool = True,
    close: bool = False,
):
    """Plot one program on the UMAP.

    Parameters
    ----------
    program
        Integer program ID from `program_activity_long_v4.csv`.
    mode
        "score" plots continuous cNMF score for the program.
        "dominant" highlights cells where this program is the dominant program.
    save
        Save files into OUT_DIR using SAVE_FORMATS.
    show
        Display the figure in the notebook.
    close
        Close the figure after display/save to reduce memory use in loops.
    """
    program = int(program)
    if program not in program_to_col:
        raise KeyError(f"Program P{program} is not mapped to a cnmf.X column.")

    label = label_map.get(program, f"P{program}")
    label_short = short_label(label)
    color = prog_color.get(program, "#377eb8")

    fig, ax = plt.subplots(figsize=FIGSIZE)

    ax.scatter(
        umap_coords[:, 0],
        umap_coords[:, 1],
        c=BACKGROUND_COLOR,
        s=BACKGROUND_SIZE,
        rasterized=True,
        linewidths=0,
        zorder=1,
    )

    if mode == "score":
        score_values = np.asarray(scores[:, program_to_col[program]], dtype=float)
        mask = np.isfinite(score_values) & (score_values > SCORE_MIN)
        n_cells = int(mask.sum())

        if n_cells > 0:
            mask_idx = np.flatnonzero(mask)
            order = mask_idx[np.argsort(score_values[mask_idx])]
            positive_scores = score_values[mask]
            vmax = float(np.quantile(positive_scores, SCORE_VMAX_QUANTILE))
            if not np.isfinite(vmax) or vmax <= SCORE_MIN:
                vmax = float(np.nanmax(positive_scores))
            if not np.isfinite(vmax) or vmax <= SCORE_MIN:
                vmax = SCORE_MIN + 1e-12

            scatter = ax.scatter(
                umap_coords[order, 0],
                umap_coords[order, 1],
                c=score_values[order],
                cmap=make_program_cmap(program),
                vmin=SCORE_MIN,
                vmax=vmax,
                s=POINT_SIZE,
                rasterized=True,
                linewidths=0,
                zorder=2,
                alpha=HIGHLIGHT_ALPHA,
            )
            cbar = fig.colorbar(scatter, ax=ax, fraction=0.045, pad=0.015)
            cbar.set_label("cNMF score", fontsize=6)
            cbar.ax.tick_params(labelsize=5, length=2)

        title_suffix = f"score > {SCORE_MIN:g}: {n_cells:,} cells"

    elif mode == "dominant":
        mask = (max_score > DOMINANT_THRESHOLD) & (dominant_program == program)
        n_cells = int(mask.sum())

        if n_cells > 0:
            ax.scatter(
                umap_coords[mask, 0],
                umap_coords[mask, 1],
                c=color,
                s=POINT_SIZE,
                rasterized=True,
                linewidths=0,
                zorder=2,
                alpha=HIGHLIGHT_ALPHA,
            )

        title_suffix = f"dominant cells > {DOMINANT_THRESHOLD:g}: {n_cells:,}"

    else:
        raise ValueError("mode must be either 'score' or 'dominant'.")

    style_umap_axis(ax)
    ax.set_title(f"P{program} {label_short}\n{title_suffix}", fontsize=7)
    plt.tight_layout()

    saved_paths = []
    if save:
        file_stem = f"umap_P{program:02d}_{safe_filename(label_short)}_{mode}"
        for file_format in SAVE_FORMATS:
            path = OUT_DIR / f"{file_stem}.{file_format}"
            fig.savefig(path, format=file_format, bbox_inches="tight")
            saved_paths.append(path)

    if show:
        plt.show()

    if close:
        plt.close(fig)

    return fig, ax, saved_paths


# Preview the first program.
plot_program(programs[0], mode=PLOT_MODE, save=False, show=True, close=False)

# %% [markdown]
# ## Optional: interactive single-program viewer
#
# This cell shows one program at a time using a dropdown. If `ipywidgets` is unavailable, use `plot_program(program_id)` directly.

# %%
try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display

    program_options = [
        (f"P{program} {short_label(label_map.get(program, f'P{program}'))}", program)
        for program in programs
    ]

    program_dropdown = widgets.Dropdown(
        options=program_options,
        value=programs[0],
        description="Program",
        layout=widgets.Layout(width="420px"),
    )
    mode_dropdown = widgets.ToggleButtons(
        options=["score", "dominant"],
        value=PLOT_MODE,
        description="Mode",
    )
    output = widgets.Output()

    def update_program_plot(*_):
        with output:
            clear_output(wait=True)
            plot_program(
                program_dropdown.value,
                mode=mode_dropdown.value,
                save=False,
                show=True,
                close=True,
            )

    program_dropdown.observe(update_program_plot, names="value")
    mode_dropdown.observe(update_program_plot, names="value")
    display(program_dropdown, mode_dropdown, output)
    update_program_plot()

except Exception as exc:
    print(f"ipywidgets viewer not available: {exc}")
    print("Use plot_program(program_id, mode='score') instead.")

# %% [markdown]
# ## Display and save every CSV-listed program one by one
#
# Running this cell will show every program in the notebook and save per-program SVG/PNG files.
# Dense point layers are rasterized inside SVG files to keep them compact; text remains editable.

# %%
RUN_ALL_PROGRAMS = True
SAVE_EACH_PROGRAM = True
DISPLAY_EACH_PROGRAM = True

all_saved_paths = []
if RUN_ALL_PROGRAMS:
    for program in programs:
        _, _, saved_paths = plot_program(
            program,
            mode=PLOT_MODE,
            save=SAVE_EACH_PROGRAM,
            show=DISPLAY_EACH_PROGRAM,
            close=True,
        )
        all_saved_paths.extend(saved_paths)

    print(f"Finished plotting {len(programs):,} programs")
    if SAVE_EACH_PROGRAM:
        print(f"Saved {len(all_saved_paths):,} files to: {OUT_DIR}")
else:
    print("RUN_ALL_PROGRAMS is False. Set it to True to display/save every program.")
